# 08 - 数据质量校验

本 Notebook 完成以下任务：
1. 全表数据量概览
2. 个股日线深度校验（缺失/价格/涨跌幅/成交量）
3. 生成质量报告

---

## 校验规则

| 检查项 | 规则 | 阈值 |
|--------|------|------|
| 缺失交易日 | 对比 trade_calendar | 允许 ≤5 天 |
| 价格合理性 | close ∈ [low, high]，无零价/负价 | 0 容忍 |
| 涨跌幅异常 | abs(pct_chg) ≤ 22% | 含 20% 涨跌停 + 2% 容差 |
| 低成交量 | volume ≥ 100 手 | 预警 |

## 1. 全表数据量概览

In [ ]:
from invest_model.db import get_engine
from invest_model.validators.data_validator import DataValidator

engine = get_engine()
validator = DataValidator(engine)

overview = validator.validate_all_tables()
print(overview.summary())

In [ ]:
# 以表格形式展示
df = overview.to_dataframe()
print(df.to_string(index=False))

## 2. 个股日线深度校验

In [ ]:
from invest_model.repositories.stock_pool_repo import StockPoolRepository

pool_repo = StockPoolRepository(engine)
stock_codes = pool_repo.get_pool_codes("core")
print(f"校验标的: {stock_codes}\n")

daily_report = validator.validate_stock_daily(stock_codes)
print(daily_report.summary())

In [ ]:
# 只看不通过的项
df = daily_report.to_dataframe()
failed = df[~df['passed']]
if failed.empty:
    print("所有校验项均通过")
else:
    print(f"有 {len(failed)} 项未通过:")
    print(failed.to_string(index=False))

## 完成

数据质量校验完毕。如有异常请检查数据源或重新采集。

继续 `09_daily_pipeline.ipynb` 配置每日自动更新管线。